# MobileBERT for the phone (Workstream G.4): data → fine-tune → calibrate → ONNX int8

Builds the level-3 second opinion the app loads from `android-app/app/src/main/assets/models/mobilebert/`
(`ondevice/MobileBert.kt`; contract in `android-app/tools/README.md`, "MobileBERT on the phone").

**Runtime → Change runtime type → T4 GPU, then Run all.** About 20–40 min. Safe to run all: it writes **only** to
`MyDrive/CSC3104/mobilebert/` (never `bert/`, never Hugging Face). Needs nothing from Drive: the repo files it uses
are embedded in cell 2.

**What it does**
1. English only (the phone's levels 2–3 are English-only, and `google/mobilebert-uncased` has an English vocab):
   BothBosu (3 sets) + the English hard calls from `notebooks/data/bert_hard_calls.jsonl` (×4 in train).
2. Windows of up to 5 caller lines built with the server's own `bert_text.make_window`; scam windows only from
   caller line 3; split by call. Never trained on: the English android scripts (incl. `scam_en_90/91`) and the
   robocall set; the G.5 benchmark tests on those.
3. Fine-tunes `google/mobilebert-uncased` in **fp32** (MobileBERT gives NaN losses in fp16), `max_len` 128,
   **left truncation** (the newest line is always read, as on the phone).
4. Fits a temperature on the validation split.
5. Writes `vocab.txt`, checks the phone's tokenizer (`tokenizers` BertWordPiece from that `vocab.txt`, which
   `WordPieceTokenizerTest` proves equal to the Kotlin one) gives the same ids as training, exports ONNX (opset 17)
   → int8, and checks int8 vs PyTorch agreement (target > 98%, as C.2).
6. Writes the 3 files the phone needs: `model.onnx`, `vocab.txt`, `bert_config.json`.

In [ ]:
# 1. Setup
!pip -q install onnx "onnxruntime==1.20.1"     # only these: upgrading datasets/tokenizers can break Colab's transformers
import os, re, sys, json, time, hashlib, random, inspect
import numpy as np, pandas as pd
from google.colab import drive
drive.mount('/content/drive')
import transformers, datasets, tokenizers, torch
print('transformers', transformers.__version__, '| datasets', datasets.__version__, '| torch', torch.__version__)

BASE = '/content/drive/MyDrive/CSC3104'
WORK = f'{BASE}/mobilebert'                # everything this notebook writes (never bert/)
OUT = f'{WORK}/model_out'                  # the 3 files for android-app/app/src/main/assets/models/mobilebert/
assert '/bert/' not in OUT + '/', 'must not write into the server BERT folder'
os.makedirs(OUT, exist_ok=True)
REPO = '/content/repo_files'               # cell 2 writes the repo files here

SEED = 42
random.seed(SEED); np.random.seed(SEED)
MODEL_NAME = 'google/mobilebert-uncased'
MAX_LEN = 128                              # tokens incl. [CLS]/[SEP]; = bert_config max_len (the phone's default)
LR, EPOCHS, BATCH = 5e-5, 4, 32            # if the loss goes NaN: LR = 3e-5 and rerun from cell 7
MAX_WINDOWS_PER_CALL = 6
FIRST_SCAM_LINE = 2                        # scam windows must END at caller line >= 3 (index 2)
HARD_REPEAT = 4                            # hard-example TRAIN windows are repeated this often
AGREE_TARGET = 0.98                        # int8 vs PyTorch: same decision at 0.5 on > 98% of windows
VERSION = 'mobilebert-v1-' + time.strftime('%Y%m%d')
print(VERSION, '->', OUT)

In [ ]:
# 2. Repo files, embedded from main (2026-11-11) so nothing is needed from Drive. Don't edit.
#    bert_text.py = backend/workers/bert/bert_text.py (the server's window function: train = serve)
#    hard_calls_en.jsonl = the English rows of notebooks/data/bert_hard_calls.jsonl
#    robocall_gold.csv = backend/workers/urgency_en/eval/robocall_gold.csv (test only)
#    scripts_en.json = caller lines of android-app/app/src/main/assets/scripts/*_en_*.json (test only)
FILES = {'bert_text.py': '"""bert_text.py - the ONE function that turns a call\'s recent caller lines into the text BERT reads.\n\nUsed by BOTH the training notebook (notebooks/bert_cascade.ipynb builds every training window with\nmake_window) and the BERT pod (predict.py), so training and serving can never prepare text differently\n(the lesson from the ZH workers\' name gap).\n\nLight normalisation only: BERT needs natural text, so no lowercasing (the model is cased), no placeholder\ntokens and no name stripping. NFKC (full-width -> normal), traditional -> simplified Chinese, whitespace\ncollapsed. Lines are joined oldest -> newest with " | "; the tokenizer truncates from the LEFT, so when the\nwindow is too long the OLDEST words are dropped and the newest line is always read.\n"""\nimport re\nimport unicodedata\n\ntry:\n    from opencc import OpenCC\n    _t2s = OpenCC("t2s").convert\nexcept ImportError:  # still runs; traditional characters are then not converted\n    _t2s = lambda s: s  # noqa: E731\n\nMAX_TURNS = 5          # caller lines per window (the gateway sends up to 5)\nJOIN = " | "           # a visible separator the tokenizer keeps ("\\n" would be dropped as whitespace)\n_ZH = re.compile(r"[一-鿿㐀-䶿]")\n_WS = re.compile(r"\\s+")\n\n\ndef normalize(line):\n    s = unicodedata.normalize("NFKC", str(line))\n    if _ZH.search(s):\n        s = _t2s(s)\n    return _WS.sub(" ", s).strip()\n\n\ndef make_window(lines, max_turns=MAX_TURNS):\n    """caller lines, oldest first -> one string. Keeps the last `max_turns` non-empty lines."""\n    lines = [normalize(x) for x in lines if str(x).strip()]\n    return JOIN.join(lines[-max_turns:])\n\n\nif __name__ == "__main__":\n    print(make_window(["您好，我是新加坡警察部隊的。", "  你的戶口涉及洗錢案  ", "", "請馬上轉賬"]))\n', 'hard_calls_en.jsonl': '{"call_id": "hard_en_legit_bank_bill_reminder", "lang": "en", "label": 0, "kind": "bank_bill_reminder", "lines": ["Hi, good afternoon, I\'m calling from the cards team at your bank about your monthly statement.", "Your statement balance this month is five hundred and twelve dollars, and the payment due date is the eighteenth.", "If you\'ve already paid, please ignore this call, it can take a day or two to show.", "You can pay through your banking app, at any AXS station, or set up GIRO so you don\'t miss it.", "We won\'t ask you for any card details or one-time password on this call.", "Alright, thank you for your time, have a nice evening."]}\n{"call_id": "hard_en_legit_bank_mortgage_reminder", "lang": "en", "label": 0, "kind": "bank_mortgage_reminder", "lines": ["Hello, may I speak to Mr Lim? I\'m calling from the home loans department.", "This is just a courtesy reminder that your mortgage instalment will be deducted on the first of next month.", "Please make sure there\'s enough balance in your linked savings account by then.", "There\'s nothing else you need to do, the deduction is automatic.", "If you\'d like to change the deduction account, you can visit any branch with your IC.", "Okay, thank you Mr Lim, goodbye."]}\n{"call_id": "hard_en_legit_bank_minimum_payment", "lang": "en", "label": 0, "kind": "bank_minimum_payment", "lines": ["Good morning, this is Rachel from the credit card collections team.", "Our records show the minimum payment for last month\'s statement hasn\'t come in yet.", "The minimum amount is fifty dollars, and a late fee may apply if it\'s not paid by Friday.", "You can make the payment yourself in the app or internet banking, whichever is easier for you.", "Please don\'t give your card number or PIN to anyone who calls, including us.", "Oh, you paid it this morning? Sorry for the trouble then, have a good day."]}\n{"call_id": "hard_en_legit_bank_fraud_alert_confirmed", "lang": "en", "label": 0, "kind": "bank_fraud_alert_confirmed", "lines": ["Hello, this is the fraud monitoring team from your bank. I\'m calling about a card transaction.", "There was an online purchase of one hundred and ninety dollars at an electronics store about an hour ago.", "Was that you? You can just answer yes or no, I don\'t need any other details.", "Okay, thanks for confirming. Since it was you, there\'s nothing more to do and your card stays active.", "For your safety, we will never ask you to transfer money or read out an OTP.", "Have a good evening."]}\n{"call_id": "hard_en_legit_bank_fraud_alert_blocked", "lang": "en", "label": 0, "kind": "bank_fraud_alert_blocked", "lines": ["Good evening, I\'m calling from your bank\'s card security team about some unusual activity.", "We stopped a payment attempt of eight hundred dollars from an overseas website this afternoon.", "If you don\'t recognise it, I can block the card now and a new one will be mailed to your registered address.", "Alright, the card is blocked. Your money is safe and the payment did not go through.", "You don\'t need to give me any codes or passwords, and you don\'t need to move any money.", "The replacement card should arrive in five to seven working days. Thank you."]}\n{"call_id": "hard_en_legit_bank_login_alert_app", "lang": "en", "label": 0, "kind": "bank_login_alert_app", "lines": ["Hi, this is a call from your bank regarding an alert on your account.", "We sent you a notification in the app asking you to confirm a new device login from this morning.", "Please open the app yourself when convenient and tap whether it was you.", "If it wasn\'t you, the app will guide you to lock your account, or you can call the hotline on our website.", "I won\'t ask for your login details or any code over the phone.", "Thank you, goodbye."]}\n{"call_id": "hard_en_legit_bank_card_posted", "lang": "en", "label": 0, "kind": "bank_card_posted", "lines": ["Hello, I\'m calling from the bank to let you know your replacement debit card has been posted.", "It should reach your mailbox within the week.", "When it arrives, you can activate it in the banking app under card services.", "Your old card will stop working once the new one is activated.", "Remember, no one from the bank will ask you for the PIN or the code we send you.", "That\'s all from me, thank you."]}\n{"call_id": "hard_en_legit_bank_branch_appointment", "lang": "en", "label": 0, "kind": "bank_branch_appointment", "lines": ["Good morning, this is the Tampines branch calling about your appointment to update your account details.", "We have you down for Thursday at two p.m. with our customer service officer.", "Please bring your NRIC and a recent proof of address, like a utility bill.", "The update usually takes about twenty minutes.", "If you need to reschedule, you can do it through the appointment page in the app.", "See you on Thursday, bye."]}\n{"call_id": "hard_en_legit_bank_scam_awareness_call", "lang": "en", "label": 0, "kind": "bank_scam_awareness_call", "lines": ["Hello, this is a short service call from your bank, it\'s about scam awareness.", "We\'re reminding customers that scammers are pretending to be bank staff and police officers.", "Real bank staff will never ask you to move money to a safe account or share your one-time password.", "If anyone asks you to do that, just hang up and call our official hotline yourself.", "There\'s nothing you need to do now, your account is fine.", "Thank you for your time and stay safe."]}\n{"call_id": "hard_en_legit_bank_phishing_followup", "lang": "en", "label": 0, "kind": "bank_phishing_followup", "lines": ["Hi, I\'m following up on the suspicious SMS you reported to us yesterday.", "Thank you for forwarding it. We\'ve confirmed it was a phishing message and not from us.", "We have reported the link so it can be taken down.", "Since you didn\'t click it or enter any details, your account is safe.", "Just a reminder, we never send links asking you to log in or update your card.", "Thanks again for reporting it. Goodbye."]}\n{"call_id": "hard_en_legit_telco_bill", "lang": "en", "label": 0, "kind": "telco_bill", "lines": ["Hello, this is customer service from your mobile operator about your latest bill.", "Your bill for this month is sixty-eight dollars and it\'s due on the fifteenth.", "It\'s a bit higher because of the roaming charges from your trip last week.", "You can see the breakdown in the app, and pay there or through GIRO.", "If you\'d like a roaming plan for next time, I can send you the details by SMS.", "Okay, thank you, have a good day."]}\n{"call_id": "hard_en_legit_telco_contract_renewal", "lang": "en", "label": 0, "kind": "telco_contract_renewal", "lines": ["Hi, good afternoon, I\'m calling from your broadband provider. Your two-year contract ends next month.", "We have a renewal offer that keeps your current speed at thirty-nine dollars a month.", "There\'s no need to decide now, you can renew in the app or at any of our shops.", "If you don\'t renew, the service just continues at the normal monthly rate.", "I\'ll send the offer to your email so you can read it first.", "Thank you, have a good evening."]}\n{"call_id": "hard_en_legit_utility_bill_overdue", "lang": "en", "label": 0, "kind": "utility_bill_overdue", "lines": ["Hello, this is the utilities customer service team calling about your electricity and water account.", "Your last bill of one hundred and twenty dollars is a few days overdue.", "We just wanted to remind you before the next bill comes, there\'s no disconnection at this stage.", "The app, internet banking or any AXS machine will all work for the payment.", "If you\'re having difficulty, we can arrange a payment plan with you.", "Alright, thank you, goodbye."]}\n{"call_id": "hard_en_legit_town_council_charges", "lang": "en", "label": 0, "kind": "town_council_charges", "lines": ["Good morning, I\'m calling from the town council office about your service and conservancy charges.", "Our system shows two months of charges outstanding, about one hundred and sixty dollars in total.", "You can pay at our office counter, by GIRO, or through the AXS machines.", "If you\'ve already arranged GIRO recently, it may not have started yet.", "Our office is at the void deck of block two-one-five, open till six.", "Okay, thank you for your time."]}\n{"call_id": "hard_en_legit_insurance_premium_due", "lang": "en", "label": 0, "kind": "insurance_premium_due", "lines": ["Hello, this is Daniel from your insurance company\'s customer care team.", "I\'m calling because the annual premium for your travel and home policy is due at the end of the month.", "The amount is two hundred and thirty dollars, the same as last year.", "You can pay through our customer portal or by cheque, the details are in the letter we mailed you.", "If you\'d like to cancel instead, just let your agent know, there\'s no penalty.", "Thank you, have a nice day."]}\n{"call_id": "hard_en_legit_insurance_claim_documents", "lang": "en", "label": 0, "kind": "insurance_claim_documents", "lines": ["Hi, I\'m calling from the claims department about the hospital claim you sent in last month.", "We still need the official hospital receipt before we can continue processing it.", "You can upload the receipt on our customer portal, or post it to the address in our letter.", "Once we have it, the result usually takes about ten working days.", "Any payout goes to the bank account you registered with us, so you don\'t need to give bank details on the phone.", "Thank you, goodbye."]}\n{"call_id": "hard_en_legit_clinic_outstanding_balance", "lang": "en", "label": 0, "kind": "clinic_outstanding_balance", "lines": ["Hello, this is the polyclinic calling about your visit last Monday.", "There\'s a small outstanding balance of eighteen dollars for the medicine you collected.", "It\'s nothing urgent, you can pay at the counter on your next visit or through the health app.", "Your Medisave and subsidy have already been applied to the bill.", "Is there anything else you\'d like to check?", "Okay, take care, goodbye."]}\n{"call_id": "hard_en_legit_dental_consultation_fee", "lang": "en", "label": 0, "kind": "dental_consultation_fee", "lines": ["Hi, I\'m calling from the dental clinic to confirm your braces consultation next Saturday.", "The consultation is eighty dollars, and you\'ll pay at the clinic after the visit.", "If you decide to go ahead with treatment, we\'ll go through the payment plan with you in person.", "Please come fifteen minutes early to fill in the forms.", "We\'re on the second floor, beside the pharmacy.", "See you then, bye."]}\n{"call_id": "hard_en_legit_courier_cash_on_delivery", "lang": "en", "label": 0, "kind": "courier_cash_on_delivery", "lines": ["Hello, I\'m the courier for the parcel you ordered online. It\'s cash on delivery.", "The amount on the slip is forty-two dollars.", "I\'ll be at your block in about twenty minutes.", "You can pay in cash, or scan the payment QR code printed on the delivery slip.", "Please make sure someone is home to collect it.", "Okay, see you soon."]}\n{"call_id": "hard_en_legit_courier_redelivery", "lang": "en", "label": 0, "kind": "courier_redelivery", "lines": ["Hi, I\'m calling from the delivery company. We tried to deliver your parcel this morning but no one was home.", "We can deliver again tomorrow between nine and twelve, is that okay?", "There\'s no extra charge for the second attempt.", "If you prefer, you can collect it from the post office near your place with your IC.", "You can also change the delivery time in the courier\'s official app.", "Alright, tomorrow morning then. Thank you."]}\n{"call_id": "hard_en_legit_landlord_rent", "lang": "en", "label": 0, "kind": "landlord_rent", "lines": ["Hi, it\'s Mr Tan, your landlord. Just a quick call about the rent.", "This month\'s rent is due on Friday as usual, same amount, same account.", "Also, the aircon servicing is on Saturday morning, the technician will come at ten.", "If anything else in the unit needs fixing, let me know and I\'ll arrange it.", "Alright, thanks, talk to you next time."]}\n{"call_id": "hard_en_legit_tuition_fee", "lang": "en", "label": 0, "kind": "tuition_fee", "lines": ["Hello, this is Ms Wong from the tuition centre, calling about your son\'s classes.", "The fees for next term are three hundred and twenty dollars, same as before.", "You can pay at our front desk or by bank transfer to the centre\'s account on the invoice.", "Payment is due by the first lesson of the term.", "By the way, he did very well in the last test, he\'s improving a lot.", "Thank you, see you on Saturday."]}\n{"call_id": "hard_en_legit_car_workshop_quote", "lang": "en", "label": 0, "kind": "car_workshop_quote", "lines": ["Hello, this is the workshop calling about your car.", "We found that the front brake pads need changing, along with the normal servicing.", "The total will be about three hundred and fifty dollars including parts and labour.", "Should we go ahead? It\'ll be ready by five this evening.", "You can pay by card or PayNow to the workshop when you collect the car.", "Okay, we\'ll call you when it\'s done."]}\n{"call_id": "hard_en_legit_gym_membership_renewal", "lang": "en", "label": 0, "kind": "gym_membership_renewal", "lines": ["Hi, I\'m calling from the gym about your membership.", "Your yearly membership ends on the thirtieth, and we have a renewal rate for existing members.", "It\'s ninety-nine dollars a month, or one thousand and fifty for the full year.", "No rush at all, you can renew at the front desk the next time you come in.", "Your locker booking will carry over if you renew.", "Thanks, see you at the gym."]}\n{"call_id": "hard_en_legit_friend_split_bill", "lang": "en", "label": 0, "kind": "friend_split_bill", "lines": ["Hey, it\'s Wei Ling. Thanks for coming to dinner last night!", "I\'ve worked out the bill, it came to two hundred and forty in total for six people.", "So it\'s forty dollars each. Send it over whenever you\'re free, no rush.", "I\'ll post the receipt photo in the group chat later.", "Also, are you joining the hike on Sunday?", "Okay, see you then!"]}\n{"call_id": "hard_en_legit_parent_sends_money", "lang": "en", "label": 0, "kind": "parent_sends_money", "lines": ["Hello boy, it\'s Papa. Did you reach home already?", "Your mother says you need money for your school trip next week.", "I\'ll transfer the hundred and fifty to your account tonight after dinner.", "Check your bank app tomorrow and tell me if you got it.", "And remember to bring your jacket, it\'s cold up there.", "Okay, sleep early."]}\n{"call_id": "hard_en_legit_catering_order_balance", "lang": "en", "label": 0, "kind": "catering_order_balance", "lines": ["Hi, this is the restaurant calling about your catering order for Saturday.", "That\'s forty pax, buffet style, delivered at eleven thirty.", "The total comes to six hundred and eighty dollars, and the deposit you paid has already been deducted.", "The balance can be paid to our staff on the day, by cash or card.", "Do you need us to bring extra plates and cutlery?", "Alright, thank you, we\'ll see you on Saturday."]}\n{"call_id": "hard_en_legit_condo_maintenance_fee", "lang": "en", "label": 0, "kind": "condo_maintenance_fee", "lines": ["Good afternoon, I\'m calling from the condo management office.", "The quarterly maintenance fees were sent out last week, and yours is still showing as unpaid.", "The amount is nine hundred and sixty dollars.", "You can pay by cheque at our office or by bank transfer, the account is printed on the notice.", "The office is open on weekdays until five thirty.", "Thank you, goodbye."]}\n{"call_id": "hard_en_legit_hr_salary_delay", "lang": "en", "label": 0, "kind": "hr_salary_delay", "lines": ["Hi, I\'m calling from HR about this month\'s salary.", "There was a delay with the bank file, so salaries will be credited on Monday instead of today.", "Everyone in the company is affected, it\'s not just your account.", "You don\'t need to do anything, and your bank details stay the same.", "Sorry for the inconvenience, we\'ll send an email to everyone as well.", "Thanks for understanding."]}\n{"call_id": "hard_en_legit_bank_overseas_card_use", "lang": "en", "label": 0, "kind": "bank_overseas_card_use", "lines": ["Hello, this is your bank\'s card services, returning your call about using your card overseas.", "Your card can already be used in Japan, you don\'t need to activate anything.", "You can also set your overseas spending limit in the app before you travel.", "If your card is lost while you\'re away, lock it in the app and call our overseas hotline.", "We\'ll never ask you for your PIN or the code from our SMS.", "Have a safe trip, goodbye."]}\n{"call_id": "hard_en_scam_family_nephew_accident", "lang": "en", "label": 1, "kind": "family_nephew_accident", "lines": ["Hello, Uncle? It\'s me, don\'t you recognise my voice?", "Yes, it\'s Marcus. I\'m using my friend\'s phone, mine is spoilt.", "Uncle, I\'m in trouble. I hit someone\'s car and they want compensation now or they\'ll make a police report.", "I need three thousand dollars today. Can you transfer it to my friend\'s account first?", "Please don\'t tell my parents, they\'ll be very angry with me.", "I\'ll return you the money next month, I promise. Please be quick."]}\n{"call_id": "hard_en_scam_family_son_hospital_overseas", "lang": "en", "label": 1, "kind": "family_son_hospital_overseas", "lines": ["Mum, it\'s me. I\'m calling from a different number because I lost my phone.", "I\'m in Bangkok, I fell down and I\'m in the hospital now.", "The hospital won\'t treat me until I pay a deposit of four thousand dollars.", "My cards are not working here. Can you send the money to the hospital agent\'s account?", "I\'ll send you the account number on WhatsApp. Please hurry, they\'re waiting.", "Don\'t tell Dad yet, I don\'t want him to worry."]}\n{"call_id": "hard_en_scam_family_grandson_bail", "lang": "en", "label": 1, "kind": "family_grandson_bail", "lines": ["Grandma, it\'s your grandson. Can you hear me?", "I\'m in big trouble, I got arrested because my friend had drugs in the car.", "The lawyer says I can go home today if we pay bail of five thousand dollars.", "Please don\'t tell anyone, I\'m so embarrassed.", "The lawyer will call you with the bank details. Please do it quickly.", "Thank you Grandma, I love you."]}\n{"call_id": "hard_en_scam_friend_stranded_overseas", "lang": "en", "label": 1, "kind": "friend_stranded_overseas", "lines": ["Hey, it\'s Kelvin, sorry to call you so suddenly.", "I\'m stuck at the airport in Jakarta, my wallet and passport were stolen.", "I need to pay for a temporary document and a new flight today, around one thousand two hundred.", "Can you lend me the money? Transfer it to this account, it\'s my cousin\'s, I can\'t log in to mine.", "I\'ll pay you back as soon as I\'m home. Please, I really have no one else to ask.", "Can you do it in the next half hour? The counter closes soon."]}\n{"call_id": "hard_en_scam_family_daughter_hostel_fee", "lang": "en", "label": 1, "kind": "family_daughter_hostel_fee", "lines": ["Hi Mummy, it\'s me, this is my new number, please save it.", "My old phone fell into the toilet so I\'m using this one now.", "Mummy, I need to pay my hostel fee today or they\'ll kick me out, it\'s eight hundred dollars.", "Can you PayNow it to this number instead? My bank app isn\'t working on the new phone.", "Please do it now, the office closes at five.", "Thanks Mummy, don\'t call the old number, it\'s not working."]}\n{"call_id": "hard_en_scam_family_nephew_friend_surgery", "lang": "en", "label": 1, "kind": "family_nephew_friend_surgery", "lines": ["Hello, Auntie Mei? This is your nephew\'s friend, Jun Hao.", "Daryl was in an accident on his motorbike, he\'s at the hospital now.", "He asked me to call you because his phone is broken. The surgery needs a deposit first.", "Can you transfer two thousand five hundred to my account? I\'ll pass it to the hospital.", "Please don\'t call his parents yet, he doesn\'t want them to panic.", "Please hurry, the doctor is waiting for the payment."]}\n{"call_id": "hard_en_scam_family_sibling_rent", "lang": "en", "label": 1, "kind": "family_sibling_rent", "lines": ["Hi Jie, it\'s me. Sorry, I\'m using a friend\'s phone.", "I need your help, my company\'s payment didn\'t come in and my rent is overdue.", "The landlord says if I don\'t pay two thousand by tonight he\'ll change the locks.", "Can you transfer it to my landlord directly? I\'ll send you his account now.", "Please don\'t mention it to Ma, I\'ll settle it myself next week.", "Thank you, please send me the screenshot after you transfer."]}\n{"call_id": "hard_en_scam_friend_classmate_loan", "lang": "en", "label": 1, "kind": "friend_classmate_loan", "lines": ["Hello, guess who? It\'s your old classmate from secondary school!", "Sorry I changed my number, I\'ve been busy with my new business.", "Actually I\'m in a tight spot, my supplier needs payment today or I lose the whole order.", "Can you lend me fifteen hundred dollars? I\'ll pay you back with interest on Friday.", "Please keep this between us, I don\'t want the others to know.", "Just transfer to this account and send me a screenshot. Thanks, you\'re a lifesaver."]}\n{"call_id": "hard_en_scam_family_son_army_equipment", "lang": "en", "label": 1, "kind": "family_son_army_equipment", "lines": ["Pa, it\'s me, I\'m using my campmate\'s phone, mine got confiscated.", "I lost a piece of equipment during the exercise and I have to pay for it today.", "It\'s one thousand eight hundred dollars, otherwise I\'ll be charged and can\'t book out.", "Can you transfer it to my sergeant\'s account? He\'ll settle it for me.", "Don\'t tell Ma, she\'ll worry for nothing.", "Please be fast, I have to return the phone soon."]}\n{"call_id": "hard_en_scam_bank_otp_reassurance", "lang": "en", "label": 1, "kind": "bank_otp_reassurance", "lines": ["Good afternoon, this is the bank\'s security team. We never ask for your password, so don\'t worry.", "We detected a login to your internet banking from a new phone in Malaysia.", "To stop it, I\'m sending you a verification code now. Please read me the six digits.", "This code is only to cancel the login, it\'s different from a password.", "Quickly please, the other person is still logged in.", "Thank you. Now please stay on the line while I secure your account."]}\n{"call_id": "hard_en_scam_bank_card_reversal_reassurance", "lang": "en", "label": 1, "kind": "bank_card_reversal_reassurance", "lines": ["Hello, I\'m calling from your credit card centre. This is not a scam call, you can check our name online.", "A purchase of two thousand nine hundred dollars is pending on your card.", "If it\'s not yours, I can reverse it, but I need you to confirm the card number and expiry date.", "Don\'t worry, we are the bank, it\'s safe to give it to me.", "Also read me the code that just came by SMS so the system can approve the reversal.", "The reversal will expire in ten minutes, so please be fast."]}\n{"call_id": "hard_en_scam_bank_safe_account_reassurance", "lang": "en", "label": 1, "kind": "bank_safe_account_reassurance", "lines": ["Hi, this is the bank\'s anti-scam unit. Remember, the bank never asks you to transfer money.", "But your account has been linked to a scam case, so your savings are at risk right now.", "We need to move your funds to a protected holding account until the investigation ends.", "This is a special procedure, so the normal rule does not apply.", "Please open your banking app and add the account I give you as a payee.", "Don\'t tell the branch staff, they are not authorised to know about this case."]}\n{"call_id": "hard_en_scam_bank_fake_app_update", "lang": "en", "label": 1, "kind": "bank_fake_app_update", "lines": ["Hello, this is customer support from your bank. We\'ll never ask for your PIN.", "We\'re upgrading security, and your account will be frozen today unless you update the app.", "Please download the new version from the link I\'m sending you by SMS, not from the app store.", "After installing, allow all permissions so the update can complete.", "Then log in once so the system can verify you.", "Thank you, your account will be safe after that."]}\n{"call_id": "hard_en_scam_bank_fee_refund_card_details", "lang": "en", "label": 1, "kind": "bank_fee_refund_card_details", "lines": ["Hi, I\'m from your bank\'s card services. We are refunding an annual fee charged by mistake.", "To process the refund of one hundred and ninety-two dollars, I need to verify your card.", "Please read out the long number on the front of your card and the security code.", "You\'ll receive a code from us now, just read it to me so the refund goes through.", "The bank never asks for your password, this is only a verification code.", "Thank you, the refund will reach you within today."]}\n{"call_id": "hard_en_scam_telco_disconnection_link", "lang": "en", "label": 1, "kind": "telco_disconnection_link", "lines": ["This is the billing department of your mobile operator. Your line will be cut off in two hours.", "There\'s an overdue amount of three hundred and eighty dollars on your account.", "To keep your number active, you must pay now through the link I\'m sending you.", "If you don\'t pay today, your number will be terminated and passed to a debt collector.", "Enter your card details on the page and the code you receive to confirm.", "Please do it while I\'m on the line."]}\n{"call_id": "hard_en_scam_utility_cutoff_paynow", "lang": "en", "label": 1, "kind": "utility_cutoff_paynow", "lines": ["Hello, I\'m calling from the utilities company. Your electricity will be disconnected this evening.", "Our records show four months of unpaid bills.", "To avoid the cut-off, you need to pay a reconnection deposit right now.", "Please transfer five hundred and sixty dollars to our collection officer\'s PayNow number.", "Our officer is already on the way to your block to cut the supply.", "Send me the screenshot once it\'s done and I\'ll cancel the job."]}\n{"call_id": "hard_en_scam_card_debt_court_threat", "lang": "en", "label": 1, "kind": "card_debt_court_threat", "lines": ["Good morning, this is the credit card recovery department.", "Your card is three months overdue and the bank is filing a court case against you today.", "You can stop the case if you settle twelve hundred dollars now.", "Transfer the payment to our settlement officer\'s account, I\'ll give you the number.", "Do not call the bank hotline, they cannot stop the legal process.", "Please confirm once you\'ve transferred."]}\n{"call_id": "hard_en_scam_post_fee_link", "lang": "en", "label": 1, "kind": "post_fee_link", "lines": ["Hi, this is the post office. Your parcel couldn\'t be delivered because the address is incomplete.", "There\'s also an unpaid handling fee of two dollars ninety.", "I\'ll send you a link, please key in your card details there to pay the fee.", "You\'ll get an OTP, enter it on the page as well.", "If it\'s not paid today, the parcel will be returned to the sender.", "Thank you, that\'s all."]}\n{"call_id": "hard_en_scam_health_refund_banking_login", "lang": "en", "label": 1, "kind": "health_refund_banking_login", "lines": ["Hello, I\'m calling from the national health insurance office.", "You were overcharged for your last hospital stay, and there\'s a refund of six hundred dollars.", "To receive it, I need your internet banking user ID and the SMS code we will send.", "This is an official refund, so please don\'t hang up.", "The refund must be claimed within today, otherwise it will be cancelled.", "Thank you, please read me the code now."]}\n{"call_id": "hard_en_scam_insurance_lapse_agent_account", "lang": "en", "label": 1, "kind": "insurance_lapse_agent_account", "lines": ["Hello, this is your insurance company. Your policy will lapse today because the last payment failed.", "You\'ll lose all your coverage, including the hospital plan.", "To reinstate it, please pay the premium now to this account, the usual portal is under maintenance.", "The account is under our agent\'s name, that\'s normal for urgent cases.", "Please send me a screenshot after you transfer.", "Do it quickly, the system closes at six."]}\n', 'robocall_gold.csv': 'id,text,label,status,source_file,category,label_time_ai\n1,So if you want to know about this case just press 1.,0,both_0,audio-wav-16khz/974387_normalized.wav,both_0,0\n2,"Thanks to your good payment history and good credit score, you have been qualified finally for an interest rate reduction between 0 to 5 percent.",0,both_0,audio-wav-16khz/314200_normalized.wav,both_0,0\n3,Your social security number has been used for some kind of fraudulent activities in the south border of Texas.,1,both_1,audio-wav-16khz/798937_normalized.wav,both_1,0\n4,Call the number on your caller ID now for this amazing deal.,1,both_1,audio-wav-16khz/269242_normalized.wav,both_1,0\n5,This is a disconnection notice from your electric company.,0,both_0,audio-wav-16khz/1361753_normalized.wav,both_0,0\n6,"Now, you should have received something in the mail about this warranty and since we haven\'t gotten a response, we are giving you a final courtesy call before we close out your file.",1,both_1,audio-wav-16khz/179182_normalized.wav,both_1,1\n7,I don\'t know what you\'re referring to.,0,both_0,audio-wav-16khz/28719_normalized.wav,both_0,0\n8,"Hi, we\'ve been trying to reach you concerning your vehicle\'s extended warranty.",0,both_0,audio-wav-16khz/179182_normalized.wav,both_0,0\n9,"Serious allegations have been filed against your name, due to which you will be taken into federal custody for further investigation.",1,both_1,audio-wav-16khz/1663081852.97905_in_Every1_Telecom_Robocall__CBP-GovtImpers-P2_normalized.wav,both_1,0\n10,"We have gone through your account, and we found that your Amazon account has been used for shopping purpose, as we can see multiple orders placed from your Amazon account.",0,disagree,audio-wav-16khz/1211775_normalized.wav,disagree,0\n11,"Placed for Apple, I fund 12 Pro using your Amazon account.",1,disagree,audio-wav-16khz/500499_normalized.wav,disagree,\n12,"Hi, my name is Jennifer Kelly.",0,both_0,audio-wav-16khz/269242_normalized.wav,both_0,0\n13,Very good day to you.,0,both_0,audio-wav-16khz/28719_normalized.wav,both_0,0\n14,Discover has made some big changes on your credit card accounts and you are now eligible for a complete debt elimination on all of your credit cards.,1,disagree,audio-wav-16khz/552425_normalized.wav,disagree,0\n15,"If you have a debit balance of your loan, you will have to resume paying back.",0,both_0,audio-wav-16khz/1226300_normalized.wav,both_0,0\n16,"And if you do not authorize this order, press 1 or 0.",1,both_1,audio-wav-16khz/795479_normalized.wav,both_1,0\n17,your case and take necessary action on this matter.,0,both_0,audio-wav-16khz/926755_normalized.wav,both_0,\n18,The reason you receive this phone call from our department is to inform you that there\'s a legal enforcement action filed on your social security number for fraudulent activities.,1,both_1,audio-wav-16khz/525134_normalized.wav,both_1,0\n19,It\'s urgent that you return my call today to complete your application.,1,both_1,audio-wav-16khz/1183793_normalized.wav,both_1,1\n20,"So if you want to cancel the payment and want to talk our customer service, please press one.",0,disagree,audio-wav-16khz/1254978_normalized.wav,disagree,0\n21,You can say things like responding to a text or returning a phone call.,0,both_0,audio-wav-16khz/360189_normalized.wav,both_0,0\n22,"We go ahead and proceed that parcel to you, sir.",0,both_0,audio-wav-16khz/28719_normalized.wav,both_0,0\n23,"This number is registered to our Amazon, and this number has been purchased an iPhone XS from our Amazon.",1,both_1,audio-wav-16khz/28719_normalized.wav,both_1,0\n24,"My state assigned phone number is 855-228-4781 and this call has been marked complete by state agent Mary Marcello, agent number 5568.",0,both_0,audio-wav-16khz/274450_normalized.wav,both_0,0\n25,"To connect the call press 1, I repeat to avoid any kind of legal consequences or a non-available arrest warrant, you need to talk to the concerned officer immediately.",1,both_1,audio-wav-16khz/1298020_normalized.wav,both_1,1\n26,The amount charged is $149.,0,both_0,audio-wav-16khz/948020_normalized.wav,both_0,0\n27,Because we are thinking of a database.,0,both_0,audio-wav-16khz/28719_normalized.wav,both_0,0\n28,"So, it\'s a reminder call to you.",0,both_0,audio-wav-16khz/28719_normalized.wav,both_0,0\n29,to your account before the next billing cycle.,0,both_0,audio-wav-16khz/27991_normalized.wav,both_0,\n30,The state has now set aside $1.3 billion.,0,both_0,audio-wav-16khz/188142_normalized.wav,both_0,0\n31,The reason of this call is to inform you about a package has been seized under your name by U.S.,1,both_1,audio-wav-16khz/453389_normalized.wav,both_1,0\n32,"The very second you receive this message you need to leave your work aside, so that we can discuss about your case and take necessary action on this matter in order to connect.",1,both_1,audio-wav-16khz/832646_normalized.wav,both_1,1\n33,Please press 1 to contact Fraud Department for dispute.,1,both_1,audio-wav-16khz/1222036_normalized.wav,both_1,0\n34,"So it will be showing you after 12 hours, sir, because the order is now going to shipping mode.",0,both_0,audio-wav-16khz/28719_normalized.wav,both_0,0\n35,Your telephone number has been pre-selected to receive a complimentary stay in one of our five-star hotels.,0,both_0,audio-wav-16khz/65627_normalized.wav,both_0,0\n36,You have been officially notified.,0,both_0,audio-wav-16khz/509700_normalized.wav,both_0,0\n37,"Since we have not gotten a response, we are giving you a final courtesy call before you close out your file.",1,both_1,audio-wav-16khz/532475_normalized.wav,both_1,1\n38,We are monitoring your payment history and you have been qualified to drop down your interest rate below 6% on your credit cards.,0,both_0,audio-wav-16khz/62991_normalized.wav,both_0,0\n39,call is to inform you about a package has been seized under your name by U.S.,1,both_1,audio-wav-16khz/453398_normalized.wav,both_1,0\n40,This has been an important call regarding legal enforcement action being executed under your name for suspicious and fraudulent activities carried out using your identity.,1,both_1,audio-wav-16khz/431150_normalized.wav,both_1,0\n41,"aside, so that we can discuss about your case and take necessary action on this matter.",1,both_1,audio-wav-16khz/719738_normalized.wav,both_1,\n42,There is one pre-authorized order placed for Apple MacBook Pro and AirPod Pro for a total amount of $1499 it will be deducted from your account soon.,1,both_1,audio-wav-16khz/1114595_normalized.wav,both_1,1\n43,your work aside so that we can discuss about your case and take necessary action on this matter.,0,both_0,audio-wav-16khz/936582_normalized.wav,both_0,\n44,"If you did not authorize this transaction, please press 1 to connect with Fraud Specialist immediately.",1,both_1,audio-wav-16khz/1222041_normalized.wav,both_1,1\n45,I see my Amazon account.,0,both_0,audio-wav-16khz/28719_normalized.wav,both_0,0\n46,payment of $469 for your recent purchase.,0,both_0,audio-wav-16khz/1239946_normalized.wav,both_0,\n47,"If you approve this transaction, please press 2.",0,both_0,audio-wav-16khz/917074_normalized.wav,both_0,0\n48,This is an effort from this administration as part of their financial relief program to help Americans recover financially from the virus outbreak.,0,both_0,audio-wav-16khz/368913_normalized.wav,both_0,0\n49,Thank you for choosing Hilton Hotels.,0,both_0,audio-wav-16khz/807777_normalized.wav,both_0,0\n50,been seized under your name by U.S.,1,both_1,audio-wav-16khz/373798_normalized.wav,both_1,\n51,You have been qualified to drop down your entry straight below 6 on your credit card.,0,disagree,audio-wav-16khz/55573_normalized.wav,disagree,0\n52,"Someone order an iPhone X from your Amazon account, which will charge you $899.",1,both_1,audio-wav-16khz/605706_normalized.wav,both_1,0\n53,You now qualify for 0% interest rate on all your credit card account.,0,both_0,audio-wav-16khz/535208_normalized.wav,both_0,0\n54,own call is to inform you that we will be suspending your social security number because we found some suspicious activities on your social security.,1,both_1,audio-wav-16khz/598178_normalized.wav,both_1,0\n55,Is that okay with you?,0,both_0,audio-wav-16khz/28719_normalized.wav,both_0,0\n56,Press 1 if you did not do this.,0,both_0,audio-wav-16khz/735436_normalized.wav,both_0,0\n57,"Let me explain to you, sir.",0,both_0,audio-wav-16khz/28719_normalized.wav,both_0,0\n58,"We have seen a recent order number, AMZ0987, of iPhone 11 Pro on your account, which is billed on your card attached to your Amazon account.",1,disagree,audio-wav-16khz/1638881759.18243_in_Family_Communication_Robocall__Amazon-SuspiciousCharge_normalized.wav,disagree,0\n59,fraudulent and suspicious activity in the state of Texas.,1,disagree,audio-wav-16khz/28740_normalized.wav,disagree,\n60,Only recommend please do not access your account on your device for security reason until your account will not get fixed.,1,disagree,audio-wav-16khz/940839_normalized.wav,disagree,0\n61,In order to connect to federal agent press one and you will be connected to the concerned department.,0,disagree,audio-wav-16khz/702986_normalized.wav,disagree,0\n62,Thank you for connecting with Amazon.,0,both_0,audio-wav-16khz/28719_normalized.wav,both_0,0\n63,"calling you from the legal department to inform you that, due to some suspicious activities related to your name, we are trying to reach you, but unfortunately we missed your answer.",1,both_1,audio-wav-16khz/509700_normalized.wav,both_1,0\n64,"Call our 800 number at 833-304-1447, UST 456.",0,both_0,audio-wav-16khz/260545_normalized.wav,both_0,0\n65,This is an emergency call from your tech support company.,1,both_1,audio-wav-16khz/1626726860.369721_in_AKA_Management_Robocall__TechSupport-Refund_normalized.wav,both_1,\n66,You need to be simply just verify that cancellation form by your site.,0,disagree,audio-wav-16khz/28719_normalized.wav,disagree,0\n67,Press 1 to get the refund.,0,both_0,audio-wav-16khz/1626726860.369721_in_AKA_Management_Robocall__TechSupport-Refund_normalized.wav,both_0,0\n68,So sorry if you\'re busy.,0,both_0,audio-wav-16khz/88408_normalized.wav,both_0,0\n69,"Hello, this is Mary Marcello regarding your COVID-19 relief payment and you can reach me at my state assigned phone number, 855-228-4781.",0,both_0,audio-wav-16khz/274458_normalized.wav,both_0,0\n70,in charge for the payment of $469 for your recent purchase.,1,disagree,audio-wav-16khz/1239951_normalized.wav,disagree,\n71,"Hi, this is Officer Sarah Schulz calling you from the Department of Social Security Administration.",1,disagree,audio-wav-16khz/1641490672.138122_in_Inteliquent_Robocall__SSA-LegalNotice_normalized.wav,disagree,0\n72,I would call soon so you don\'t miss the cutoff.,0,disagree,audio-wav-16khz/368913_normalized.wav,disagree,1\n73,Press 2 to be put on our do not call list or press 1 to speak to someone about possibly extending or reinstating your car\'s warranty.,0,disagree,audio-wav-16khz/70587_normalized.wav,disagree,0\n74,We strongly recommend please don\'t access your account on your device for security reason until your account will not get fixed for more support.,1,both_1,audio-wav-16khz/817218_normalized.wav,both_1,0\n75,"Hello, this is Alex with American Benefits on a recorded line.",0,both_0,audio-wav-16khz/1221882_left.wav,both_0,0\n76,You are now qualified for a complete debt elimination on all of your credit cards.,1,disagree,audio-wav-16khz/1263033_normalized.wav,disagree,0\n77,your electric service will be disconnected within the next 30 minutes due to non-payment.,1,disagree,audio-wav-16khz/1361739_normalized.wav,disagree,1\n78,Which computer you are using right now at the moment?,1,disagree,audio-wav-16khz/28719_normalized.wav,disagree,0\n79,"And it\'s my job to make you aware that because you qualified for a relief payment, you also qualify for the new debt relief program.",0,both_0,audio-wav-16khz/274458_normalized.wav,both_0,0\n80,And we just suspended your social security number.,1,both_1,audio-wav-16khz/241125_normalized.wav,both_1,0\n81,I repeat press 1 to get more details.,1,disagree,audio-wav-16khz/387224_normalized.wav,disagree,0\n82,"This is the Notifications Department with Jersey Central Power and Light, you have an automatic interruption of service in the next 30 to 45 minutes due to request a discontinue of the service or delinquent account status.",1,both_1,audio-wav-16khz/61836_normalized.wav,both_1,1\n83,a disconnection of service today within 30 minutes due to non-payment on the account.,1,both_1,audio-wav-16khz/567528_normalized.wav,both_1,1\n84,"Hi, this is an automated message from the Disconnection Department of Pacific Gas and Electric, Electricity Company.",0,both_0,audio-wav-16khz/s6diZGlyX3I3cGRpcjp0b21jYXQ5MzE1OjE2MzQ4NDA2ODIyMTNL98qFNd.gen_Global_Net_Holdings_Robocall__Utility-30MinDisconnect_normalized.wav,both_0,0\n85,"resident, you are now able to take advantage of alternative federal student loan repayment options and hardship programs.",0,disagree,audio-wav-16khz/360172_normalized.wav,disagree,0\n86,"Again, this is the Notifications Department with Jersey Central Power and Light, you have an automatic interruption of service in the next 30 to 45 minutes due to request a discontinue of the service or delinquent account status.",1,both_1,audio-wav-16khz/61836_normalized.wav,both_1,1\n87,Or to talk to our customer care please press 3.,0,both_0,audio-wav-16khz/1366379_normalized.wav,both_0,0\n88,In order to connect our legal department press 1 so that we can discuss about your case file and take necessary action on this matter.,1,both_1,audio-wav-16khz/509676_normalized.wav,both_1,0\n89,you that there is an order placed for Apple iPhone 11 Pro using your Amazon account.,1,both_1,audio-wav-16khz/549504_normalized.wav,both_1,0\n90,"Now, you should have received something in the mail about that warranty.",0,both_0,audio-wav-16khz/102399_normalized.wav,both_0,0\n91,This call is regarding your purchase of $999 for iPhone 13 Pro Max.,1,both_1,audio-wav-16khz/638999_normalized.wav,both_1,0\n92,I look forward to speaking with you today.,0,both_0,audio-wav-16khz/s6diZGlyXzRiOWRpcjp0b21jYXQ2NDkzOjE2MTkxOTM4NTkxMTRBTZDSgD.gen_DID_Central_Robocall__StudentLoan_-_New_Changes_normalized.wav,both_0,0\n93,Drop down your injure straight below 6 on your credit card.,0,disagree,audio-wav-16khz/57715_normalized.wav,disagree,\n94,I repeat press 1 to get more information.,1,disagree,audio-wav-16khz/440547_normalized.wav,disagree,0\n95,We encounter a serious issue coming out of your computer.,1,both_1,audio-wav-16khz/1631035650.66108_in_AKA_Management_Robocall__Computer-Hijack_normalized.wav,both_1,0\n96,"And if you don\'t respond, Bank of America will cancel this promotion.",1,both_1,audio-wav-16khz/1246811_normalized.wav,both_1,0\n97,"On your excellent payment history, you now qualify for a 0% interest rate on all your credit card accounts.",0,both_0,audio-wav-16khz/520125_normalized.wav,both_0,0\n98,"of PSE&G New Jersey, Electricity Company.",0,both_0,audio-wav-16khz/567525_normalized.wav,both_0,\n99,FedEx International Express will inform you for the last time that you have a parcel that has been delivered twice and no one has signed for it.,1,both_1,audio-wav-16khz/1195631_normalized.wav,both_1,1\n100,"Now is your chance to get into a program so that you can qualify for the $10,000 debt elimination.",1,both_1,audio-wav-16khz/1226300_normalized.wav,both_1,\n101,"by third party supplier, you will be receiving $50 reward along with 30% discount on your electric and gas bill, please press 1 and get your $50 reward card.",1,both_1,audio-wav-16khz/73738_normalized.wav,both_1,0\n102,"If you are interested in renewing your auto warranty now, please press 5 now or press 9 to be removed from our list.",1,disagree,audio-wav-16khz/568965_normalized.wav,disagree,0\n103,To repeat this message please press 4.,0,both_0,audio-wav-16khz/1366379_normalized.wav,both_0,0\n104,"If you find this to be an error or would like to stop the interruption, please press 1 to speak with the billing department.",0,both_0,audio-wav-16khz/1205449_normalized.wav,both_0,0\n105,By this call you have been officially notified.,0,both_0,audio-wav-16khz/509676_normalized.wav,both_0,0\n106,citizens on our state list that have tax debt about the new zero-tax owed tax debt relief program that was recently put into effect by the Biden administration and is open for enrollment.,0,disagree,audio-wav-16khz/1629378389.587442_in_Peerless_Robocall__DebtRelief-ZeroTax_normalized.wav,disagree,0\n107,From suspicious activity from your account we have found an unauthorized login attempt.,1,both_1,audio-wav-16khz/817196_normalized.wav,both_1,0\n108,Department of the Social Security Administration.,1,both_1,audio-wav-16khz/Social_Security_Scam_-_Call_the_SSA_Call_20210122T194442Z_IDT_telecom_normalized.wav,both_1,0\n109,It\'s a Visa card What\'s a Visa card?,0,both_0,audio-wav-16khz/28549_normalized.wav,both_0,0\n110,If you do not authorize this order to cancel press 1.,0,disagree,audio-wav-16khz/608000_normalized.wav,disagree,0\n111,The reason you have received this phone call from our department is to inform you that there\'s legal enforcement actions filed on your social security number for fraudulent activities.,1,both_1,audio-wav-16khz/28736_normalized.wav,both_1,0\n112,"Have a great day, sir.",0,both_0,audio-wav-16khz/28719_normalized.wav,both_0,0\n113,My direct phone number is 714-406-4370.,0,both_0,audio-wav-16khz/s6diZGlyXzRiOWRpcjp0b21jYXQ2NDkzOjE2MTkxOTM4NTkxMTRBTZDSgD.gen_DID_Central_Robocall__StudentLoan_-_New_Changes_normalized.wav,both_0,0\n114,"Hello, this is Sarah from Discover.",0,both_0,audio-wav-16khz/1630350028.16156_in_Range_Telecom_Robocall__CreditCard-RateReduction_normalized.wav,both_0,0\n115,"Well, what can I do about it?",0,both_0,audio-wav-16khz/28719_normalized.wav,both_0,0\n116,"Otherwise, please press 2 to speak with the billing department.",0,both_0,audio-wav-16khz/1175828_normalized.wav,both_0,0\n117,We\'ve been trying to reach you concerning your car\'s extended warranty.,0,both_0,audio-wav-16khz/65173_normalized.wav,both_0,0\n118,"The very second you receive this message, you need to leave your work aside so that we can discuss about your case and take necessary action on this matter in order to connect.",1,both_1,audio-wav-16khz/832638_normalized.wav,both_1,1\n119,"So please in order, go ahead, get more information to speak with officer press one.",0,both_0,audio-wav-16khz/560647_normalized.wav,both_0,0\n120,"suspicious, and fraudulent activities carried out using your identities.",1,both_1,audio-wav-16khz/482002_normalized.wav,both_1,\n121,"that your social security number has been used for some kind of fraudulent activities in the south border of Texas, so please in order go ahead get more information.",1,both_1,audio-wav-16khz/825247_normalized.wav,both_1,0\n122,"Due to non-payment on the account, please press 1 to speak with a representative.",0,disagree,audio-wav-16khz/s6diZGlyX3dqbWRpcjp0b21jYXQxMzY1OjE2MzIzNTIyNTc0NDg1Yft6tv.gen_MatrixTelecom_Robocall__Utility-30MinDisconnect_normalized.wav,disagree,0\n123,"Greetings, this is an automated message from the Disconnection Department of the Marva Power and Light to inform you that you will be experiencing a disconnection of service today within 30 minutes.",1,both_1,audio-wav-16khz/520014_normalized.wav,both_1,1\n124,"Hello, this call is from Microsoft Help Center.",0,both_0,audio-wav-16khz/1631035650.66108_in_AKA_Management_Robocall__Computer-Hijack_normalized.wav,both_0,0\n125,Press 1 now to speak to our qualification department and complete the enrollment process.,1,disagree,audio-wav-16khz/520125_normalized.wav,disagree,0\n126,"Using our automated approval technology, you are now able to obtain enrollment information based on your current financial status.",0,both_0,audio-wav-16khz/307201_normalized.wav,both_0,0\n127,It\'s my job to remind you.,0,both_0,audio-wav-16khz/28719_normalized.wav,both_0,0\n128,Press 1 and you will be connected to the concerned CBP officer.,0,both_0,audio-wav-16khz/562333_normalized.wav,both_0,0\n129,The reason you have received this phone call from our department is to inform you that there\'s a legal enforcement actions filed on your name for fraudulent activities.,1,both_1,audio-wav-16khz/694449_normalized.wav,both_1,0\n130,"Due to some suspicious activities related to your social security number, we are forced to suspend your social security number with immediate effect.",1,both_1,audio-wav-16khz/s6diZGlyX3BiNWRpcjp0b21jYXQ0NzY3OjE2MTkxMTc0ODUzOTd652aDjV.gen_iBasis_Robocall__SSA-P1-BenefitsCanceled__GovtImpers__normalized.wav,both_1,0\n131,"So when you get this message, kindly press 1 to connect with the next available officer.",0,both_0,audio-wav-16khz/537336_normalized.wav,both_0,0\n132,that we will be suspending your social security number because we found some suspicious activities on your social security.,1,both_1,audio-wav-16khz/787192_normalized.wav,both_1,0\n133,receive this message you need to leave your work aside so that we can discuss about your case and take necessary action on this matter in order to connect.,1,both_1,audio-wav-16khz/826868_normalized.wav,both_1,1\n134,"By this call, you have been officially notified.",0,both_0,audio-wav-16khz/509708_normalized.wav,both_0,0\n135,"If you find this to be an error or would like to stop the disjoin, please press 1 to start your claim, otherwise press 2 to speak with the billing department.",1,disagree,audio-wav-16khz/1157297_normalized.wav,disagree,0\n136,To speak with Officer press 1.,0,both_0,audio-wav-16khz/530083_normalized.wav,both_0,0\n137,found some suspicious activity in your Amazon account.,1,disagree,audio-wav-16khz/698028_normalized.wav,disagree,0\n138,"This was just passed by the Biden administration, actively known as BidenCare.",0,both_0,audio-wav-16khz/274452_normalized.wav,both_0,0\n139,"Department of Social Security Administration, the reason you have received this phone call from our department is to inform you that there\'s a legal enforcement action held on your social security number for fraudulent activities.",1,both_1,audio-wav-16khz/28872_normalized.wav,both_1,0\n140,"For more information, press 1 and hold for the next developmental officer.",0,both_0,audio-wav-16khz/544713_normalized.wav,both_0,0\n141,"You have a parcel that has been delivered twice, and no one has signed for it.",0,disagree,audio-wav-16khz/1195647_normalized.wav,disagree,0\n142,"My Capital One card Capital One, I got you.",0,disagree,audio-wav-16khz/28549_normalized.wav,disagree,0\n143,This call is also recorded online with our security purpose.,1,disagree,audio-wav-16khz/28719_normalized.wav,disagree,0\n144,"This call is all about your parcel, sir.",0,both_0,audio-wav-16khz/28719_normalized.wav,both_0,0\n145,With contraband of drugs and cash.,1,both_1,audio-wav-16khz/488973_normalized.wav,both_1,\n146,"If you would like to keep coverage or extend it, press 8 to speak to a customer service agent and go over options.",0,both_0,audio-wav-16khz/927202.wav,both_0,0\n147,Press two to be put on our do not call list.,0,both_0,audio-wav-16khz/64023_normalized.wav,both_0,0\n148,If it\'s not fixed right away then your computer will become obsolete and all of your credential information may got compromised.,1,both_1,audio-wav-16khz/1631035650.66108_in_AKA_Management_Robocall__Computer-Hijack_normalized.wav,both_1,1\n149,"If you did not authorize this order, then press 1 for Amazon support.",0,disagree,audio-wav-16khz/635443_normalized.wav,disagree,0\n150,"After two failed attempts, I am calling you for the last time.",1,both_1,audio-wav-16khz/601882_normalized.wav,both_1,1\n151,"below 6 on your credit card, so please press 1 to speak with live operator and get the confirmation number.",0,both_0,audio-wav-16khz/132842_normalized.wav,both_0,0\n152,"So sorry if you\'re busy, I just had some fantastic news.",0,both_0,audio-wav-16khz/s6diZGlyXzRiOWRpcjp0b21jYXQ2NDkzOjE2MTkxOTM4NTkxMTRBTZDSgD.gen_DID_Central_Robocall__StudentLoan_-_New_Changes_normalized.wav,both_0,0\n153,"And thank you for connecting, sir.",0,both_0,audio-wav-16khz/28719_normalized.wav,both_0,0\n154,"And you can reach me at my state-assigned phone number, 844-877-0123.",0,both_0,audio-wav-16khz/274460_normalized.wav,both_0,0\n155,Nadine Smith calling in from the processing department.,0,both_0,audio-wav-16khz/s6diZGlyXzRiOWRpcjp0b21jYXQ2NDkzOjE2MTkxOTM4NTkxMTRBTZDSgD.gen_DID_Central_Robocall__StudentLoan_-_New_Changes_normalized.wav,both_0,0\n156,"Customer, this is Molly from Amazon.",0,both_0,audio-wav-16khz/s6diZGlyX3p6d2RmYTp0b21jYXQ2MzQwOjE2NDkzNzI3ODQ4MDNNxF8yiu.gen_TelecomFirm_Robocall__Amazon-SuspiciousCharge-P1_normalized.wav,both_0,0\n157,"A petition is set to be filed on your behalf to extend the suspension of payments indefinitely, meaning you would have a permanent zero-dollar balance.",1,both_1,audio-wav-16khz/28561_normalized.wav,both_1,0\n158,Press 4 if you want to be added in DNC list.,0,both_0,audio-wav-16khz/1626726860.369721_in_AKA_Management_Robocall__TechSupport-Refund_normalized.wav,both_0,0\n159,that you have been charged for the payment of $469 for your recent purchase.,1,both_1,audio-wav-16khz/1239948_normalized.wav,both_1,0\n160,11 Pro using your Amazon account.,0,both_0,audio-wav-16khz/549491_normalized.wav,both_0,\n161,"Dear customer, this is David from Amazon.",0,both_0,audio-wav-16khz/800146_normalized.wav,both_0,0\n162,you to let you know that your social security number has been used for some kind of fraudulent activities in the south border of Texas.,1,both_1,audio-wav-16khz/701106_normalized.wav,both_1,0\n163,"You will be charged $600, sir.",1,both_1,audio-wav-16khz/28719_normalized.wav,both_1,0\n164,"So you want to cancel that parcel and stop that payment, right, sir?",1,disagree,audio-wav-16khz/28719_normalized.wav,disagree,0\n165,This is an automated message from the Disconnection Department of Dominion Energy Electricity Company to inform you that you will be experiencing a disconnection of service today within 30 minutes due to non-payment on the account.,1,both_1,audio-wav-16khz/310553_normalized.wav,both_1,1\n166,"I repeat, press 1 and do not apply for it.",1,disagree,audio-wav-16khz/1246782_normalized.wav,disagree,0\n167,So we have put in hold to this transaction.,0,both_0,audio-wav-16khz/1638881759.18243_in_Family_Communication_Robocall__Amazon-SuspiciousCharge_normalized.wav,both_0,0\n168,"Instead of talking to you, I will simply go ahead and run away.",1,disagree,audio-wav-16khz/28719_normalized.wav,disagree,0\n169,"If you do not authorize this order, press 1 to speak to our representative, or press 2 to authorize.",0,both_0,audio-wav-16khz/857375_normalized.wav,both_0,0\n170,And when is the Capital One card going to be expired?,0,disagree,audio-wav-16khz/28549_normalized.wav,disagree,0\n171,activity in the state of Texas.,0,both_0,audio-wav-16khz/28743_normalized.wav,both_0,\n172,"This is your number, sir, 4 0 8 8 8 4 1 4 4 5.",0,both_0,audio-wav-16khz/28719_normalized.wav,both_0,0\n173,So when you get this message kindly press one to connect with the next available officer.,0,both_0,audio-wav-16khz/630197_normalized.wav,both_0,0\n174,The very second you read this message you need to leave your work aside so that we can discuss about your case and take necessary action.,1,both_1,audio-wav-16khz/776734_normalized.wav,both_1,1\n175,We would like to inform you that your telephone number was qualified by our booking system to receive an all-inclusive complimentary stay.,0,both_0,audio-wav-16khz/135556_normalized.wav,both_0,0\n176,Call your car at 800-833-304-1447.,0,both_0,audio-wav-16khz/58339_normalized.wav,both_0,0\n177,"And as you know, there have been some pretty big changes to the federal student loan programs recently.",0,both_0,audio-wav-16khz/1134649_normalized.wav,both_0,0\n178,"Since the student loan payment delay is about to come to an end, you can reach us directly to discuss your options at 833-631-7911.",0,both_0,audio-wav-16khz/660786_normalized.wav,both_0,0\n179,"Otherwise, please press 2 to speak with the disconnection department.",0,both_0,audio-wav-16khz/64732_normalized.wav,both_0,0\n180,"To confirm this order, please press 1, or to cancel, please press 2.",0,both_0,audio-wav-16khz/838914_normalized.wav,both_0,0\n181,"And the purpose of the call, we are trying to reach you to let you know that your Social Security number has been used for some kind of fraudulent activities in the south border of Texas.",1,both_1,audio-wav-16khz/560658_normalized.wav,both_1,0\n182,We would like to inform you that this order has been placed from your account.,0,both_0,audio-wav-16khz/864704_normalized.wav,both_0,0\n183,Please be aware that your electric service will be disconnected within the next 30 minutes due to non-payment.,1,both_1,audio-wav-16khz/1361753_normalized.wav,both_1,1\n184,Thank you for choosing PSE&G New Jersey.,0,both_0,audio-wav-16khz/567525_normalized.wav,both_0,0\n185,"In order to avail the discount, kindly call us back at 888-604-0734 from 8am to 5pm Pacific Standard Time.",0,both_0,audio-wav-16khz/872731_normalized.wav,both_0,0\n186,"Dear customer, this is a disconnection notice from your electric company.",1,both_1,audio-wav-16khz/1360309_normalized.wav,both_1,0\n187,"Power and Light, you have an automatic interruption of service in the next 30 to 45 minutes due to request a discontinue of the service or delinquent account status.",1,both_1,audio-wav-16khz/1031581_normalized.wav,both_1,1\n188,Press one and you will be connected to the concerned department.,0,both_0,audio-wav-16khz/956746_normalized.wav,both_0,0\n189,"In order to connect federal agents, press 1 and you will be connected to the concerned department.",0,both_0,audio-wav-16khz/936582_normalized.wav,both_0,0\n190,"of fraudulent activities in the south border of Texas, so please in order go ahead get more information.",1,both_1,audio-wav-16khz/833937_normalized.wav,both_1,\n191,"Well, I don\'t see that iPhone in my Amazon orders.",0,both_0,audio-wav-16khz/28719_normalized.wav,both_0,0\n192,"Dear customer, thank you for your purchase from Amazon Shopping.",0,both_0,audio-wav-16khz/1331474_normalized.wav,both_0,0\n193,We\'ve been trying to reach you as your eligibility for rate reduction is about to expire.,1,both_1,audio-wav-16khz/1630350028.16156_in_Range_Telecom_Robocall__CreditCard-RateReduction_normalized.wav,both_1,1\n194,"This is a payment confirmation call for your order ID, AMC 284U49, for a refurbishment book Pro 12, worth $497, for which your card, attached with your Amazon account, has been charged.",1,both_1,audio-wav-16khz/557666_normalized.wav,both_1,0\n195,"Sir, I need your opinion, sir, because the money is going to your account.",1,both_1,audio-wav-16khz/28719_normalized.wav,both_1,\n196,Our department is to inform you that there is a legal enforcement action filed on your social security number for fraudulent activities.,1,both_1,audio-wav-16khz/525141_normalized.wav,both_1,0\n197,Press 1 to speak with a live operator and get the confirmation code.,0,disagree,audio-wav-16khz/53078_normalized.wav,disagree,0\n198,"If you haven\'t missed payment on the bill for the last three months, get a 25% discount.",0,both_0,audio-wav-16khz/240203_normalized.wav,both_0,0\n199,with Wells Fargo is on hold due to suspicious activities.,1,disagree,audio-wav-16khz/1246782_normalized.wav,disagree,0\n200,"Well, DirecTV is offering 50% discount to all of its existing customers.",0,both_0,audio-wav-16khz/547592_normalized.wav,both_0,0\n', 'scripts_en.json': '{\n "normal_en_10_bank_reminder": {\n  "label": 0,\n  "lines": [\n   "Good morning, this is a reminder call from your bank\'s card services.",\n   "Not at all. Your credit card payment of two hundred and forty dollars is due on the twenty-fifth.",\n   "You can pay through the mobile app, internet banking, or GIRO if you have set it up.",\n   "Sure. And please remember, the bank will never ask you for your OTP or password over the phone.",\n   "No problem. If you have any questions, call the number printed on the back of your card.",\n   "Thank you, have a good day."\n  ]\n },\n "normal_en_11_clinic_appointment": {\n  "label": 0,\n  "lines": [\n   "Hello, this is the clinic calling to confirm your appointment tomorrow at ten thirty.",\n   "Great. Please bring your NRIC and any medication you are currently taking.",\n   "Yes, no food for eight hours before the blood test, but you can drink plain water.",\n   "About thirty minutes. Please arrive ten minutes early to register at the counter.",\n   "Yes, there is a carpark behind the building. The consultation fee can be paid at the counter after your visit.",\n   "See you tomorrow. Take care."\n  ]\n },\n "normal_en_12_delivery_driver": {\n  "label": 0,\n  "lines": [\n   "Hi, I\'m the delivery driver. I\'m downstairs at your block now with your order.",\n   "I\'m at the lobby near the letterboxes. Blue jacket.",\n   "No problem. Can you confirm the unit number? The app shows number twelve.",\n   "The total is twenty-three dollars fifty. It\'s cash on delivery, or you can PayNow me.",\n   "Yes, no problem."\n  ]\n },\n "normal_en_13_insurance_sales": {\n  "label": 0,\n  "lines": [\n   "Hello, good afternoon. My name is Amelia, I\'m calling from a financial advisory firm. Do you have a minute?",\n   "We have a new hospitalisation plan for young working adults. I would like to explain the benefits briefly.",\n   "That\'s great. Many people find employer coverage ends when they change jobs, so a personal plan can be a good backup.",\n   "It starts from about thirty dollars a month, depending on your age and the ward type.",\n   "Of course. This promotion ends at the end of the month, but there\'s no obligation. May I send you the brochure by email?",\n   "Understood, thank you for your time."\n  ]\n },\n "normal_en_14_family_call": {\n  "label": 0,\n  "lines": [\n   "Hello, it\'s Mum. Are you free this weekend?",\n   "Ah Ma\'s birthday is on Sunday. Your aunt booked the restaurant for twelve o\'clock.",\n   "Can you pick up the cake on the way? I\'ll PayNow you back after dinner.",\n   "The one near your office. Please remember to ask them to write her name on it.",\n   "Come by eleven thirty, so we can all walk in together.",\n   "Bye bye. Take care and eat properly ah."\n  ]\n },\n "scam_en_10_gov_official": {\n  "label": 1,\n  "lines": [\n   "Good afternoon, this is the fraud prevention unit calling about your bank account.",\n   "We detected a credit card opened under your NRIC that was used for money laundering.",\n   "I understand. Because this is a serious case, I will transfer you to the Singapore Police Force now.",\n   "I am Inspector Tan from the Singapore Police Force, badge number two zero four seven.",\n   "Your name is linked to a criminal case and you may be arrested today if you do not cooperate.",\n   "You must not tell anyone, not your family and not your bank. This investigation is confidential.",\n   "To prove your innocence, we need to check your funds. Transfer your savings into a safety account that we provide.",\n   "Do it right now, within the next thirty minutes, or a warrant will be issued.",\n   "Open your banking app and send all the money to this account number. Read me the OTP when the bank sends it.",\n   "Stay on the line. Do not hang up until the transfer is done."\n  ]\n },\n "scam_en_11_bank_otp": {\n  "label": 1,\n  "lines": [\n   "Hello, this is the security department of your bank. Is this the account holder?",\n   "We noticed a suspicious payment of three thousand two hundred dollars from your account to an overseas merchant.",\n   "Don\'t worry, I can cancel it, but I need to verify your identity quickly.",\n   "The transaction will go through in five minutes, so we must act immediately.",\n   "I am sending a six-digit code to your phone now. Please read it out to me.",\n   "Yes, that is normal, it is part of the cancellation process. Just tell me the numbers.",\n   "Also give me your card number and the three digits at the back so I can block it."\n  ]\n },\n "scam_en_12_parcel_customs": {\n  "label": 1,\n  "lines": [\n   "Hello, this is the courier service. We have a parcel under your name that is held at customs.",\n   "It was sent from overseas and the declared contents do not match, so customs has flagged it.",\n   "I will connect you to the customs enforcement officer. Please hold.",\n   "Good afternoon, I am Officer Lim from Singapore Customs. You are being investigated for importing prohibited items.",\n   "If you do not resolve this today, we will refer your case to the police and the parcel will be destroyed.",\n   "You need to pay a release fee of four hundred and eighty dollars through PayNow.",\n   "I will send you the number now. Pay within one hour, otherwise the fine will double.",\n   "After you pay, take a screenshot of the receipt and send it to me on WhatsApp."\n  ]\n },\n "scam_en_13_tech_support": {\n  "label": 1,\n  "lines": [\n   "Good morning, I am calling from the technical support team of your internet provider.",\n   "Yes, our system shows your router has been hacked and your computer is sending out viruses.",\n   "It is very serious. Hackers may already have access to your personal files.",\n   "Please turn on your computer and open the web browser. I will guide you to download a support app.",\n   "Yes, give me the nine-digit code on the screen so I can connect and clean your computer.",\n   "Now I can see your screen. The virus has reached your banking. Please log in to your internet banking so I can check.",\n   "Yes, we must move your money to a protected account before the hackers take it. Please do it right now.",\n   "The cleaning service also costs two hundred dollars. Please pay with gift cards from any convenience store."\n  ]\n },\n "scam_en_14_friend_in_trouble": {\n  "label": 1,\n  "lines": [\n   "Hello Auntie, it\'s me! Guess who this is?",\n   "Yes yes, it\'s Jason! Sorry, I changed my number, my old phone fell into the water.",\n   "Auntie, I\'m in a bit of trouble. I was in a car accident and the police are holding me at the station.",\n   "I\'m fine, but I need to pay the other driver\'s repair cost and bail today or they will charge me.",\n   "Five thousand five hundred dollars. Please don\'t tell Mum, she will be so worried.",\n   "Can you PayNow it now? I\'ll give you the number. It\'s really urgent, they are waiting.",\n   "Thank you Auntie, I\'ll pay you back next week, I promise. Please hurry."\n  ]\n },\n "scam_en_90_new_scam_test_A_report_this": {\n  "label": 1,\n  "lines": [\n   "Good evening, our recruiter saw your profile and has a work from home role for you.",\n   "You just rate products on our platform and earn commission each day.",\n   "The first task needs a small top up in USDT which comes back with your commission.",\n   "I will add you to the Telegram group where the mentor shows you the steps.",\n   "Top up again today to unlock the next task or your commission stays frozen.",\n   "Slots fill up fast so please decide tonight."\n  ]\n },\n "scam_en_91_new_scam_test_B_same_scam_new_words": {\n  "label": 1,\n  "lines": [\n   "Hi, our hiring team found your resume and we have a part time online job for you.",\n   "All you do is rate products on the platform and your commission builds up.",\n   "Task one needs a small USDT top up, which is returned together with your commission.",\n   "Our mentor in the Telegram group explains each step.",\n   "One more top up today unlocks the next task, if not your commission stays frozen.",\n   "Places are filling up fast, so decide tonight."\n  ]\n }\n}'}
os.makedirs(REPO, exist_ok=True)
for name, text in FILES.items():
    open(f'{REPO}/{name}', 'w', encoding='utf-8').write(text)
sys.path.insert(0, REPO)
import bert_text                            # the SAME text prep the server pod uses
print(sorted(os.listdir(REPO)), '|', bert_text.make_window(['Hello,   this is the bank.', 'Read me the OTP.']))

In [ ]:
# 3. Conversations -> caller lines (same parsing as bert_cascade.ipynb, English part)
SENT_END = re.compile(r'(?<=[。！？!?；;])\s*')

def turn_lines(turn, max_chars=120):
    t = re.sub(r'\s+', ' ', str(turn).replace('\\n', ' ')).strip()
    if not t:
        return []
    if len(t) <= max_chars:
        return [t]
    out, buf = [], ''
    for s in SENT_END.split(t):
        if buf and len(buf) + len(s) > max_chars:
            out.append(buf); buf = ''
        buf += s
    return out + ([buf] if buf else [])

EN_SPK = re.compile(r'(?i)(?:^|\s)(caller|receiver|suspect|innocent|scammer|victim|agent|customer|person ?[ab12])\s*:')
CALLER_TAGS = {'caller', 'suspect', 'scammer', 'agent', 'person a', 'persona', 'person 1', 'person1'}
def en_dialogue_lines(dialogue):
    parts = EN_SPK.split(str(dialogue))
    tags = [t.lower() for t in parts[1::2]]
    if not tags:
        return []
    caller = next((t for t in tags if t in CALLER_TAGS), tags[0])   # who rings first is the caller
    return [l for t, txt in zip(tags, parts[2::2]) if t == caller for l in turn_lines(txt, 300)]

def bothbosu_calls(names=('BothBosu/scam-dialogue', 'BothBosu/single-agent-scam-conversations',
                          'BothBosu/multi-agent-scam-conversation')):
    from datasets import load_dataset
    calls = []
    for name in names:
        try:
            ds = load_dataset(name)
        except Exception as e:
            print(f'skip {name}: {type(e).__name__}'); continue
        for split in ds:
            d = ds[split].to_pandas()
            text_col = next(c for c in ('dialogue', 'conversation', 'text') if c in d)
            label_col = next(c for c in ('labels', 'label', 'is_scam') if c in d)
            ok = 0
            for i, r in d.iterrows():
                lines = en_dialogue_lines(r[text_col])
                if len(lines) >= 2:
                    ok += 1
                    cid = f"{name.split('/')[1]}_{split}_{i}"
                    calls.append(dict(call_id=cid, group=cid, source=name.split('/')[1], lang='en',
                                      label=int(r[label_col]), kind=str(r.get('type', '')), lines=lines))
            print(f'{name} [{split}]: {ok}/{len(d)} dialogues parsed')
            assert ok > 0.8 * len(d), f'{name}: speaker tags not recognised, print one dialogue and fix EN_SPK'
    seen, out = set(), []                     # the same dialogue can appear in several BothBosu sets
    for c in calls:
        key = ' '.join(c['lines'])[:200]
        if key not in seen:
            seen.add(key); out.append(c)
    assert out, 'no BothBosu data: check the Colab internet connection'
    return out

def hard_calls(path):
    calls = []
    for line in open(path, encoding='utf-8'):
        r = json.loads(line)
        if r['lang'] == 'en':
            calls.append(dict(call_id=r['call_id'], group=r['call_id'], source='hard_en', lang='en',
                              label=int(r['label']), kind=r['kind'], lines=r['lines']))
    print(f'hard examples (EN): {len(calls)} calls, scams {sum(c["label"] for c in calls)}')
    return calls

In [ ]:
# 4. Load the calls; drop any call that contains a line of the phone's test material (scripts / robocall set)
calls_hard = hard_calls(f'{REPO}/hard_calls_en.jsonl')
calls = pd.DataFrame(bothbosu_calls() + calls_hard)
calls = calls[calls.lines.map(len) >= 2].reset_index(drop=True)

_key = lambda s: re.sub(r'[^a-z0-9]+', ' ', bert_text.normalize(s).lower()).strip()
ROBO = pd.read_csv(f'{REPO}/robocall_gold.csv')
SCRIPTS = json.load(open(f'{REPO}/scripts_en.json'))
held_out = {_key(t) for t in ROBO.text} | {_key(l) for s in SCRIPTS.values() for l in s['lines']}
leak = calls.lines.map(lambda ls: any(_key(l) in held_out for l in ls))
print(f'calls containing a held-out test line: {int(leak.sum())} (dropped)')
calls = calls[~leak].reset_index(drop=True)
print(calls.groupby(['source', 'label']).size().to_string())

In [ ]:
# 5. Windows (bert_text.make_window, the server's own function) + split by call
def windows_for_call(c, max_w=MAX_WINDOWS_PER_CALL, first_scam=FIRST_SCAM_LINE):
    lines = c['lines']
    ends = [i for i in range(len(lines)) if c['label'] == 0 or i >= first_scam]
    if len(ends) > max_w:
        ends = sorted({ends[k] for k in np.linspace(0, len(ends) - 1, max_w).round().astype(int)})
    out = []
    for i in ends:
        w = bert_text.make_window(lines[: i + 1])
        if len(w.split()) >= 5:
            out.append(dict(text=w, label=c['label'], lang=c['lang'], source=c['source'], group=c['group'],
                            call_id=c['call_id'], end=i))
    return out

def split_groups(calls, seed=SEED):
    from sklearn.model_selection import GroupShuffleSplit
    g = calls.group.values
    tr, rest = next(GroupShuffleSplit(1, test_size=0.2, random_state=seed).split(calls, groups=g))
    rest_calls = calls.iloc[rest]
    va, te = next(GroupShuffleSplit(1, test_size=0.5, random_state=seed).split(rest_calls, groups=rest_calls.group.values))
    split = pd.Series('train', index=calls.index)
    split.iloc[rest[va]] = 'val'; split.iloc[rest[te]] = 'test'
    return split

calls['split'] = split_groups(calls)
W = pd.DataFrame([w for c in calls.to_dict('records') for w in windows_for_call(c)])
W = W.merge(calls[['call_id', 'split']], on='call_id')
order = {'train': 0, 'val': 1, 'test': 2}           # exact duplicates across splits would leak: keep the first
W = W.sort_values('split', key=lambda s: s.map(order)).drop_duplicates('text').reset_index(drop=True)
assert not set(W[W.split == 'train'].group) & set(W[W.split != 'train'].group), 'group leak'

tr = W[W.split == 'train']
is_hard = tr.source.str.startswith('hard_')
train = pd.concat([tr[~is_hard]] + [tr[is_hard]] * HARD_REPEAT).sample(frac=1, random_state=SEED).reset_index(drop=True)
val, test = W[W.split == 'val'].reset_index(drop=True), W[W.split == 'test'].reset_index(drop=True)
print(pd.crosstab([W.split, W.source], W.label, margins=True).to_string())
print(f'\ntrain {len(train)} windows (positives {train.label.mean():.0%}), val {len(val)}, test {len(test)}')

In [ ]:
# 7. Fine-tune (fp32: MobileBERT's activations overflow in fp16)
import torch
from datasets import Dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer,
                          DataCollatorWithPadding, set_seed)
from sklearn.metrics import roc_auc_score, f1_score, fbeta_score, precision_score, recall_score
set_seed(SEED)
assert torch.cuda.is_available(), 'Runtime -> Change runtime type -> T4 GPU'
tok = AutoTokenizer.from_pretrained(MODEL_NAME)
tok.truncation_side = 'left'                    # keep the NEWEST lines when a window is too long (phone does the same)
assert tok('HELLO There')['input_ids'] == tok('hello there')['input_ids'], 'tokenizer is not uncased'
lens = [len(tok(t)['input_ids']) for t in train.text.sample(min(2000, len(train)), random_state=0)]
print('token length p50/p95/p99:', np.percentile(lens, [50, 95, 99]).round(), f'(> {MAX_LEN} are cut from the left)')

def to_ds(df):
    return Dataset.from_pandas(df[['text', 'label']].reset_index(drop=True)).map(
        lambda b: tok(b['text'], truncation=True, max_length=MAX_LEN), batched=True, remove_columns=['text'])

def softmax_scam(logits, T=1.0):
    z = np.asarray(logits, dtype=np.float64) / T
    z = np.exp(z - z.max(axis=1, keepdims=True))
    return z[:, 1] / z.sum(axis=1)

def metrics(p):
    prob = softmax_scam(p.predictions); y = p.label_ids
    return {'auc': roc_auc_score(y, prob), 'f1': f1_score(y, prob >= 0.5), 'f2': fbeta_score(y, prob >= 0.5, beta=2)}

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
args = TrainingArguments(output_dir='/content/ckpt', eval_strategy='epoch', save_strategy='epoch',
                         learning_rate=LR, per_device_train_batch_size=BATCH, per_device_eval_batch_size=64,
                         num_train_epochs=EPOCHS, warmup_steps=int(0.1 * EPOCHS * -(-len(train) // BATCH)), weight_decay=0.01, fp16=False,
                         load_best_model_at_end=True, metric_for_best_model='auc', save_total_limit=1,
                         logging_steps=50, report_to='none', seed=SEED)
trainer = Trainer(model=model, args=args, train_dataset=to_ds(train), eval_dataset=to_ds(val),
                  data_collator=DataCollatorWithPadding(tok), compute_metrics=metrics)
t0 = time.time(); trainer.train(); print(f'trained in {(time.time() - t0) / 60:.1f} min')
losses = [h['loss'] for h in trainer.state.log_history if 'loss' in h]
evals = [(h['epoch'], round(h['eval_auc'], 4), round(h['eval_loss'], 4)) for h in trainer.state.log_history if 'eval_auc' in h]
print('val per epoch (epoch, AUC, loss):', evals)
assert losses and np.all(np.isfinite(losses)) and np.isfinite(evals[-1][2]), 'NaN loss: set LR = 3e-5 in cell 1, rerun cell 1 then cell 7'
if max(a for _, a, _ in evals) < 0.9:
    print('!!! val AUC < 0.9: MobileBERT did not learn well. Try LR = 1e-4 in cell 1, rerun cell 1 then cell 7.')
trainer.save_model(f'{WORK}/hf'); tok.save_pretrained(f'{WORK}/hf')

In [ ]:
# 8. Calibrate (temperature on val) + evaluate
from scipy.optimize import minimize_scalar

def fit_temperature(logits, y):
    def nll(logT):
        p = np.clip(softmax_scam(logits, np.exp(logT)), 1e-7, 1 - 1e-7)
        return -np.mean(y * np.log(p) + (1 - y) * np.log(1 - p))
    return float(np.exp(minimize_scalar(nll, bounds=(-3, 6), method='bounded').x))

def logits_of(df):
    return trainer.predict(to_ds(df)).predictions

brier = lambda p, y: float(np.mean((p - y) ** 2))
L_val = logits_of(val)
T = fit_temperature(L_val, val.label.values)
assert T < 400, f"T = {T:.0f} hit the search limit: send this output to Claude"
print(f'temperature T = {T:.3f} | val Brier {brier(softmax_scam(L_val), val.label.values):.4f} -> '
      f'{brier(softmax_scam(L_val, T), val.label.values):.4f}')

def report(name, y, p, thr=0.5):
    yhat = p >= thr
    return dict(set=name, n=len(y), pos=int(y.sum()), auc=round(roc_auc_score(y, p), 3) if 0 < y.sum() < len(y) else None,
                P=round(precision_score(y, yhat, zero_division=0), 3), R=round(recall_score(y, yhat, zero_division=0), 3),
                F1=round(f1_score(y, yhat, zero_division=0), 3), F2=round(fbeta_score(y, yhat, beta=2, zero_division=0), 3),
                brier=round(brier(p, y), 4))

rows = [report('val', val.label.values, softmax_scam(L_val, T))]
p_test = softmax_scam(logits_of(test), T)
rows.append(report('test (all)', test.label.values, p_test))
for src in sorted(test.source.unique()):
    m = (test.source == src).values
    rows.append(report(f'test: {src}', test.label.values[m], p_test[m]))
# single sentences: the EN robocall set (label = urgency worker's "urgent", NOT scam; never trained on)
robo_df = pd.DataFrame(dict(text=[bert_text.make_window([t]) for t in ROBO.text], label=ROBO.label.astype(int)))
p_robo = softmax_scam(logits_of(robo_df), T)
rows.append(report('EN robocall sentences (urgency label)', robo_df.label.values, p_robo))
results = pd.DataFrame(rows)
print(results.to_string(index=False))
results.to_csv(f'{WORK}/results_mobilebert.csv', index=False)

In [ ]:
# 9. The app's English demo scripts (never trained on), scored as the phone asks: last 5 caller lines at each line.
# The phone asks from caller line 3 on. Goal: normal scripts stay < 0.5, scam scripts reach >= 0.5.
# scam_en_90/91 are H's learning-loop scams that every model is meant to miss until retrained.
S_rows, S_txt = [], []
for name, s in SCRIPTS.items():
    for i in range(len(s['lines'])):
        S_txt.append(dict(text=bert_text.make_window(s['lines'][: i + 1]), label=s['label'], name=name, line=i + 1))
S_df = pd.DataFrame(S_txt)
S_df['p'] = softmax_scam(logits_of(S_df), T)
for name, d in S_df.groupby('name', sort=False):
    asked = d[d.line >= 3]
    worst = asked.p.max()
    ok = (worst >= 0.5) if d.label.iloc[0] == 1 else (worst < 0.5)
    S_rows.append(dict(script=name, scam=int(d.label.iloc[0]), max_from_line3=round(worst, 3), ok=bool(ok)))
    print(f"{'OK  ' if ok else 'MISS'} {name:52s} " + ' '.join(f'{p:.2f}' for p in d.p))
S_res = pd.DataFrame(S_rows)
core = S_res[~S_res.script.str.contains('_9[01]_')]
print(f'\n{int(core.ok.sum())} of {len(core)} original scripts right (+ {int(S_res.ok.sum() - core.ok.sum())} of '
      f'{len(S_res) - len(core)} of H\'s new scams)')
S_res.to_csv(f'{WORK}/scripts_mobilebert.csv', index=False)

In [ ]:
# 10. vocab.txt + phone-tokenizer check + ONNX (opset 17) -> int8 + int8 vs PyTorch agreement
import onnx, onnxruntime as ort
from onnxruntime.quantization import quantize_dynamic, QuantType
from tokenizers import BertWordPieceTokenizer

# vocab.txt: one token per line, line number = id (what WordPieceTokenizer.fromVocabLines reads)
vocab = sorted(tok.get_vocab().items(), key=lambda kv: kv[1])
assert [i for _, i in vocab] == list(range(len(vocab))), 'vocab ids are not 0..n-1'
assert all('\n' not in t and t == t.strip() for t, _ in vocab)
with open(f'{OUT}/vocab.txt', 'w', encoding='utf-8') as f:
    f.write(''.join(t + '\n' for t, _ in vocab))
for special in ('[CLS]', '[SEP]', '[UNK]'):
    assert special in tok.get_vocab()

# The phone's tokenizer = tokenizers' BertWordPiece from that vocab.txt, lowercase, left truncation
# (WordPieceTokenizerTest proves the Kotlin tokenizer gives these exact ids). It must equal the training tokenizer.
phone_tok = BertWordPieceTokenizer(f'{OUT}/vocab.txt', lowercase=True)
phone_tok._tokenizer.enable_truncation(max_length=MAX_LEN, direction='left')
phone_tok._tokenizer.no_padding()
phone_ids = lambda text: phone_tok.encode(text).ids
check = pd.concat([val.text, test.text, robo_df.text, S_df.text]).tolist()
bad = [t for t in check if phone_ids(t) != tok(t, truncation=True, max_length=MAX_LEN)['input_ids']]
print(f'phone tokenizer vs training tokenizer: {len(check) - len(bad)} of {len(check)} windows identical')
assert not bad, f'tokenizers differ, e.g. {bad[0][:120]!r}'

class LogitsOnly(torch.nn.Module):
    def __init__(self, m):
        super().__init__(); self.m = m
    def forward(self, input_ids, attention_mask):
        return self.m(input_ids=input_ids, attention_mask=attention_mask).logits

cpu_model = AutoModelForSequenceClassification.from_pretrained(f'{WORK}/hf').eval().cpu()
dummy = tok(['hello this is a test | second line'], return_tensors='pt')
fp32 = '/content/mobilebert_fp32.onnx'
kw = {'dynamo': False} if 'dynamo' in inspect.signature(torch.onnx.export).parameters else {}
torch.onnx.export(LogitsOnly(cpu_model), (dummy['input_ids'], dummy['attention_mask']), fp32,
                  input_names=['input_ids', 'attention_mask'], output_names=['logits'],
                  dynamic_axes={'input_ids': {0: 'batch', 1: 'seq'}, 'attention_mask': {0: 'batch', 1: 'seq'},
                                'logits': {0: 'batch'}}, opset_version=17, **kw)

def ort_session(path):
    o = ort.SessionOptions(); o.intra_op_num_threads = 2
    return ort.InferenceSession(path, o, providers=['CPUExecutionProvider'])

def phone_scores(sess, texts):
    """Exactly what MobileBert.score does: ids from vocab.txt, mask all 1, batch 1, softmax(logits / T)[1]."""
    names = {i.name for i in sess.get_inputs()}
    out = []
    for t in texts:
        ids = np.array([phone_ids(t)], np.int64)
        feed = {'input_ids': ids, 'attention_mask': np.ones_like(ids)}
        if 'token_type_ids' in names:
            feed['token_type_ids'] = np.zeros_like(ids)
        out.append(softmax_scam(sess.run(None, feed)[0], T)[0])
    return np.array(out)

# agreement set: every val + test window (as C.2) up to 1,500, plus all script windows
agree = pd.concat([val, test]).sample(min(1500, len(val) + len(test)), random_state=0).reset_index(drop=True)
p_torch = softmax_scam(logits_of(agree), T)
p_fp32 = phone_scores(ort_session(fp32), agree.text)
print(f'fp32 ONNX vs PyTorch: max |diff| {np.abs(p_fp32 - p_torch).max():.4f}  (export check)')
assert np.abs(p_fp32 - p_torch).max() < 0.01, 'the fp32 export does not match PyTorch'

best = None
for qname, qkw in [('int8', {}), ('int8 per-channel', {'per_channel': True}),
                    ('int8 per-channel reduced', {'per_channel': True, 'reduce_range': True})]:
    path = f'/content/mobilebert_{qname.replace(" ", "_")}.onnx'
    quantize_dynamic(fp32, path, weight_type=QuantType.QInt8, **qkw)
    p_q = phone_scores(ort_session(path), agree.text)
    r = dict(name=qname, path=path, agree05=float(((p_q >= .5) == (p_torch >= .5)).mean()),
             agree06=float(((p_q >= .6) == (p_torch >= .6)).mean()), mean_diff=float(np.abs(p_q - p_torch).mean()),
             max_diff=float(np.abs(p_q - p_torch).max()), auc_torch=roc_auc_score(agree.label, p_torch),
             auc_q=roc_auc_score(agree.label, p_q), mb=os.path.getsize(path) / 1e6)
    print(f"{qname}: {r['mb']:.1f} MB | same decision at 0.5: {r['agree05']:.1%}, at 0.6: {r['agree06']:.1%} | "
          f"|diff| mean {r['mean_diff']:.3f} max {r['max_diff']:.3f} | AUC torch {r['auc_torch']:.3f} int8 {r['auc_q']:.3f}")
    if best is None or r['agree05'] > best['agree05']:
        best = r
    if r['agree05'] >= AGREE_TARGET and abs(r['auc_q'] - r['auc_torch']) < 0.01:
        break
if best['agree05'] < AGREE_TARGET:
    print(f"\n!!! WARNING: best int8 agreement {best['agree05']:.1%} < {AGREE_TARGET:.0%}. Shipping it anyway; report it.")

m = onnx.load(best['path'])
if m.ir_version > 9:
    m.ir_version = 9                              # ONNX Runtime Android 1.20 reads IR <= 10; 9 is safe
opsets = {o.domain or 'ai.onnx': o.version for o in m.opset_import}
assert opsets.get('ai.onnx', 0) <= 21, f'opset too new for ONNX Runtime Android 1.20: {opsets}'
onnx.save(m, f'{OUT}/model.onnx')
sess = ort_session(f'{OUT}/model.onnx')
print('inputs', [(i.name, i.type, i.shape) for i in sess.get_inputs()], '| outputs', [(o.name, o.shape) for o in sess.get_outputs()],
      '| opsets', opsets, '| IR', m.ir_version)
assert sess.get_outputs()[0].name == 'logits'
p_ship = phone_scores(sess, S_df.text)
assert np.abs(p_ship - phone_scores(ort_session(best['path']), S_df.text)).max() < 1e-6
ts = []
for t in agree.text[:200]:
    t0 = time.perf_counter(); phone_scores(sess, [t]); ts.append((time.perf_counter() - t0) * 1000)
print(f'CPU latency (2 threads, Colab CPU): p50 {np.median(ts):.0f} ms, p95 {np.percentile(ts, 95):.0f} ms')

In [ ]:
# 11. bert_config.json + hand-over
def sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1 << 20), b''):
            h.update(chunk)
    return h.hexdigest()

S_df['p_int8'] = p_ship
S_int8 = S_df[S_df.line >= 3].groupby('name', sort=False).agg(label=('label', 'first'), max_p=('p_int8', 'max'))
S_int8['ok'] = np.where(S_int8.label == 1, S_int8.max_p >= 0.5, S_int8.max_p < 0.5)
cfg = dict(version=VERSION, max_len=MAX_LEN, temperature=round(T, 4), scam_index=1, max_turns=bert_text.MAX_TURNS,
           lower_case=True, model_file='model.onnx',
           base_model=MODEL_NAME, quantization=best['name'], model_sha256=sha256(f'{OUT}/model.onnx'),
           int8_agreement_05=round(best['agree05'], 4), int8_agreement_06=round(best['agree06'], 4),
           trained_on=sorted(train.source.unique().tolist()), train_windows=int(len(train)),
           first_scam_line=FIRST_SCAM_LINE + 1, hard_repeat=HARD_REPEAT, lr=LR, epochs=EPOCHS,
           metrics=results.to_dict('records'), date=time.strftime('%Y-%m-%d'))
json.dump(cfg, open(f'{OUT}/bert_config.json', 'w'), indent=2)

print('int8, as the phone scores them (max score from caller line 3):')
print(S_int8.round(3).to_string())
print('\nSEND THIS BLOCK TO CLAUDE ' + '=' * 60)
print(results.to_string(index=False))
print(f"T = {T:.4f} | quantization {best['name']} | agreement at 0.5: {best['agree05']:.1%}, at 0.6: {best['agree06']:.1%} | "
      f"AUC torch {best['auc_torch']:.3f} vs int8 {best['auc_q']:.3f} | p50 {np.median(ts):.0f} ms")
core = S_int8[~S_int8.index.str.contains('_9[01]_')]
print(f'scripts (int8): {int(core.ok.sum())} of {len(core)} original right, H\'s new scams caught: '
      f'{int(S_int8[S_int8.index.str.contains("_9[01]_")].ok.sum())} of 2')
for f in sorted(os.listdir(OUT)):
    print(f'{f:18s} {os.path.getsize(f"{OUT}/{f}") / 1e6:7.2f} MB')
print('=' * 86)
print(f'\nDownload these 3 files from Drive {OUT}/ into android-app/app/src/main/assets/models/mobilebert/')